<a href="https://colab.research.google.com/github/MaxMariusJacobs/Smart-Blind-Cane/blob/main/yolo_gehweg_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="JlQ2tiTxqxfCxBOqB5ER") # Ersetze dies durch deinen Key
project = rf.workspace("u22programcontest").project("sidewalk-segmentation-ij8uy")
version = project.version(1)
dataset = version.download("yolov11") # Lädt die Annotationen im YOLOv11-Segmentierungsformat

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 95.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to Sidewalk-Segmentation-1 in yolov11:: 100%|██████████| 3861/3861 [00:00<00:00, 6636.81it/s]


In [ ]:
import os
import glob
import yaml

# 1. Definieren des Mappings von alt auf neu:
# Alt: 0: Roadway, 1: Sidewalk, 2: downstairs, 3: path, 4: sidewalk, 5: upstairs
# Neu: 0: sidewalk, 1: roadway, 2: path, 3: stairs
class_mapping = {
    1: 0,  # Sidewalk -> sidewalk
    4: 0,  # sidewalk -> sidewalk
    0: 1,  # Roadway  -> roadway
    3: 2,  # path     -> path
    2: 3,  # downstairs -> stairs
    5: 3   # upstairs   -> stairs
}

new_names = ['sidewalk', 'roadway', 'path', 'stairs']

# 2. Alle .txt-Labeldateien (Train, Valid, Test) umschreiben
label_files = glob.glob(f"{dataset.location}/**/labels/*.txt", recursive=True)

for file_path in label_files:
    with open(file_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_cls = int(parts[0])
        if old_cls in class_mapping:
            parts[0] = str(class_mapping[old_cls])
            new_lines.append(" ".join(parts) + "\n")

    with open(file_path, 'w') as f:
        f.writelines(new_lines)

# 3. data.yaml mit bereinigter Klassenliste aktualisieren
yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['names'] = new_names
data_cfg['nc'] = len(new_names)

with open(yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"Bereinigung abgeschlossen! Neue Klassen: {new_names}")

Bereinigung abgeschlossen! Neue Klassen: ['sidewalk', 'roadway', 'path', 'stairs']


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=320,
    batch=16,
    device=0,
    save=True
)

Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Sidewalk-Segmentation-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=6

In [ ]:
import os
import glob
from google.colab import files
from ultralytics import YOLO

# 1. Automatisch den neuesten 'best.pt' Checkpoint finden (z. B. aus train-2)
checkpoints = glob.glob("runs/segment/**/weights/best.pt", recursive=True)
if not checkpoints:
    raise FileNotFoundError("Kein Checkpoint unter runs/segment/ gefunden.")

# Neueste Datei nach Änderungsdatum wählen
latest_checkpoint = max(checkpoints, key=os.path.getmtime)
print(f"Lade neuesten Checkpoint: {latest_checkpoint}")
model = YOLO(latest_checkpoint)

# 2. Export ohne inkompatibles half=True:
# - simplify=True: Operator-Fusion via ONNXSlim
# - nms=False: 100% GPU-Ausführung ohne CPU-Fallback
# - Standard FP32: Wird auf der Smartphone-GPU nativ mit FP16-Shader-Kernen ausgeführt
export_path = model.export(
    format="tflite",
    imgsz=320,
    simplify=True,
    nms=False
)

# 3. Exportierte .tflite-Datei lokalisieren und herunterladen
tflite_files = glob.glob("runs/segment/**/weights/**/*.tflite", recursive=True) + glob.glob("runs/segment/**/weights/*.tflite", recursive=True)

if tflite_files:
    target_file = max(tflite_files, key=os.path.getmtime)
    print(f"Export erfolgreich! Lade herunter: {target_file}")
    files.download(target_file)
else:
    print(f"Datei manuell herunterladen von: {export_path}")

Lade neuesten Checkpoint: runs/segment/train-2/weights/best.pt
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n-seg summary (fused): 114 layers, 2,835,348 parameters, 0 gradients, 2.4 GFLOPs

PyTorch: starting from 'runs/segment/train-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) ((1, 40, 2100), (1, 32, 80, 80)) (5.7 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 571ms
Prepared 15 packages in 4.22s
Uninstalled 3 packages in 23ms


invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:00) [START] Write Model to runs/segment/train-2/weights/best.tflite

(00:00) [ DONE] Write Model to runs/segment/train-2/weights/best.tflite (+00:00)

LiteRT: export success ✅ 24.6s, saved as 'runs/segment/train-2/weights/best.tflite' (11.1 MB)

Export complete (24.8s)
Results saved to /content/runs/segment/train-2/weights/best.tflite
Predict:         yolo predict task=segment model=runs/segment/train-2/weights/best.tflite imgsz=320 
Validate:        yolo val task=segment model=runs/segment/train-2/weights/best.tflite imgsz=320 data=/content/Sidewalk-Segmentation-1/data.yaml  
Visualize:       https://netron.app
Export erfolgreich! Lade herunter: runs/segment/train-2/weights/best.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>